In [20]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from config.spark import spark


In [21]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,TimestampType,DoubleType

orders_schema = StructType([
    StructField("order_id",StringType()),
    StructField("customer_id",StringType()),
    StructField("order_status",StringType()),
    StructField("order_purchase_timestamp",TimestampType()),
    StructField("order_approved_at",TimestampType()),
    StructField("order_delivered_carrier_date",TimestampType()),
    StructField("order_delivered_customer_date",TimestampType()),
    StructField("order_estimated_delivery_date",TimestampType()),
])


In [22]:
orders_df = (
    spark.read.format('csv')
    .option('header','True')
   # .option('inferschema','True')
   .schema(orders_schema)
    .load('../landing/files/olist_orders_dataset.csv')
    
)

In [23]:
from pyspark.sql.functions import current_timestamp

orders_df = orders_df.withColumn("ingestion_time",current_timestamp())

In [24]:
orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- ingestion_time: timestamp (nullable = false)



In [25]:
orders_df.createOrReplaceTempView('orders')

In [26]:
spark.sql(
    """
    select * from orders
    """
).show(100)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|      ingestion_time|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|2026-10-17 13:50:...|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 1